In [59]:
import pandas as pd
import numpy as np
import plotly.express as px

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.impute import SimpleImputer
from sklearn.metrics import root_mean_squared_error, r2_score
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import Lasso

# Import dataset

In [60]:
DATA_PATH = "../data/outputs/Walmart_Store_sales_ml_output.csv"
df = pd.read_csv(DATA_PATH)

In [61]:
df.head()

,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment,Year,Month,Quarter,Week,Is_Year_End,Holiday_Flag_missing
0,6,2011-02-18,1572117.54,0,59.61,3.045,214.777523,6.858,2011.0,2.0,1.0,7.0,0,1
1,13,2011-03-25,1807545.43,0,42.38,3.435,128.616064,7.470,2011.0,3.0,1.0,12.0,0,0
2,17,2012-07-27,NaN,0,NaN,NaN,130.719581,5.936,2012.0,7.0,3.0,30.0,0,0
3,11,NaN,1244390.03,0,84.57,NaN,214.556497,7.346,NaN,NaN,NaN,NaN,0,0
4,6,2010-05-28,1644470.66,0,78.89,2.759,212.412888,7.092,2010.0,5.0,2.0,21.0,0,0


In [62]:
df.shape

(150, 14)

In [63]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 14 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Store                 150 non-null    int64  
 1   Date                  132 non-null    object 
 2   Weekly_Sales          136 non-null    float64
 3   Holiday_Flag          150 non-null    int64  
 4   Temperature           132 non-null    float64
 5   Fuel_Price            136 non-null    float64
 6   CPI                   138 non-null    float64
 7   Unemployment          135 non-null    float64
 8   Year                  132 non-null    float64
 9   Month                 132 non-null    float64
 10  Quarter               132 non-null    float64
 11  Week                  132 non-null    float64
 12  Is_Year_End           150 non-null    int64  
 13  Holiday_Flag_missing  150 non-null    int64  
dtypes: float64(9), int64(4), object(1)
memory usage: 16.5+ KB


In [64]:
df.describe(include="all")

,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment,Year,Month,Quarter,Week,Is_Year_End,Holiday_Flag_missing
count,150.000000,132,1.360000e+02,150.000000,132.000000,136.000000,138.000000,135.000000,132.000000,132.000000,132.000000,132.000000,150.000000,150.000000
unique,NaN,85,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,NaN,2012-10-19,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,NaN,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,9.866667,NaN,1.249536e+06,0.073333,61.398106,3.320853,179.898509,7.598430,2010.856061,6.393939,2.454545,25.681818,0.066667,0.080000
std,6.231191,NaN,6.474630e+05,0.261556,18.378901,0.478149,40.274956,1.577173,0.811488,3.214370,1.058327,14.001809,0.250279,0.272202
min,1.000000,NaN,2.689290e+05,0.000000,18.790000,2.514000,126.111903,5.143000,2010.000000,1.000000,1.000000,1.000000,0.000000,0.000000
25%,4.000000,NaN,6.050757e+05,0.000000,45.587500,2.852250,131.970831,6.597500,2010.000000,4.000000,2.000000,15.000000,0.000000,0.000000
50%,9.000000,NaN,1.261424e+06,0.000000,62.985000,3.451000,197.908893,7.470000,2011.000000,6.000000,2.000000,25.000000,0.000000,0.000000
75%,15.750000,NaN,1.806386e+06,0.000000,76.345000,3.706250,214.934616,8.150000,2012.000000,9.000000,3.000000,36.250000,0.000000,0.000000


# Preprocessing

In [65]:
# Date
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')

# Drop des lignes avec target manquante
df = df.dropna(subset=["Weekly_Sales"]).copy()

In [66]:
# Feature engineering Date
df['Year'] = df['Date'].dt.year.astype("Int64")
df['Month'] = df['Date'].dt.month.astype("Int64")
df["Quarter"] = df["Date"].dt.quarter.astype("Int64")
df['Week'] = df['Date'].dt.isocalendar().week.astype("Int64")
df["Is_Year_End"] = (df["Date"].dt.month == 12).astype(int)
df = df.sort_values(["Store","Date"])
df["Holiday_prev1"] = df.groupby("Store")["Holiday_Flag"].shift(1).fillna(0).astype(int)
df["Holiday_next1"] = df.groupby("Store")["Holiday_Flag"].shift(-1).fillna(0).astype(int)

# Suppression de la date
df = df.drop(columns=["Date"])

In [67]:
# Outliers sur Temperature, Fuel_Price, CPI, Unemployment
outlier_cols = ["Temperature", "Fuel_Price", "CPI", "Unemployment"]
outlier_cols = [c for c in outlier_cols if c in df.columns]

for c in outlier_cols:
    mu, sigma = df[c].mean(), df[c].std()
    lo, hi = mu - 3*sigma, mu + 3*sigma
    df = df[(df[c] >= lo) & (df[c] <= hi)]

In [68]:
df.shape

(90, 15)

In [69]:
df.head()

,Store,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment,Year,Month,Quarter,Week,Is_Year_End,Holiday_Flag_missing,Holiday_prev1,Holiday_next1
44,1,1641957.44,1,38.51,2.548,211.242170,8.106,2010,2,1,6,0,0,0,0
95,1,1494251.50,0,74.78,2.854,210.337426,7.808,2010,5,2,19,0,0,1,0
73,1,1449142.92,0,85.22,2.619,211.567306,7.787,2010,8,3,34,0,1,0,0
48,1,1624383.75,0,91.65,3.684,215.544618,7.962,2011,8,3,31,0,1,0,0
78,1,1539483.70,0,62.25,3.308,218.220509,7.866,2011,11,4,46,0,0,0,0


# Pipeline

In [70]:
y = df["Weekly_Sales"]
X = df.drop(columns=["Weekly_Sales"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

cat_cols = [c for c in ["Store", "Holiday_Flag"] if c in X_train.columns]
num_cols = [c for c in X_train.columns if c not in cat_cols]

numeric_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, num_cols),
        ("cat", categorical_pipe, cat_cols),
    ],
    remainder="drop"
)

## Baseline

## Modèle régularisé (Ridge + GridSearchCV)

In [71]:
ridge_pipe = Pipeline(steps=[
    ("preprocess", preprocess),
    ("reg", Ridge())
])

param_grid = {
    "reg__alpha": np.logspace(-3, 4, 20)
}

gs_ridge = GridSearchCV(
    ridge_pipe,
    param_grid=param_grid,
    cv=5,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

gs_ridge.fit(X_train, y_train)

best_ridge = gs_ridge.best_estimator_
pred_train = best_ridge.predict(X_train)
pred_test  = best_ridge.predict(X_test)

print("Best alpha:", gs_ridge.best_params_["reg__alpha"])
print(f"RMSE train: {root_mean_squared_error(y_train, pred_train):,.0f} | RMSE test: {root_mean_squared_error(y_test, pred_test):,.0f}")
print(f"R² train: {r2_score(y_train, pred_train):.3f} | R² test: {r2_score(y_test, pred_test):.3f}")


Best alpha: 0.06951927961775606
RMSE train: 79,496 | RMSE test: 138,791
R² train: 0.985 | R² test: 0.953


In [72]:
pred_test_ridge = best_ridge.predict(X_test)

df_pred = pd.DataFrame({
    "y_true": y_test,
    "y_pred": pred_test_ridge
})

fig = px.scatter(
    df_pred,
    x="y_true",
    y="y_pred",
    title="Ridge (test) : y_true vs y_pred",
    opacity=0.6
)

# Diagonale parfaite y = x
m = df_pred["y_true"].min()
M = df_pred["y_true"].max()
fig.add_shape(type="line", x0=m, y0=m, x1=M, y1=M)

fig.show()

In [73]:
res = pd.DataFrame(gs_ridge.cv_results_)
res["rmse_cv"] = -res["mean_test_score"]

fig = px.line(
    res,
    x="param_reg__alpha",
    y="rmse_cv",
    markers=True,
    title="Ridge : RMSE CV selon alpha",
    labels={"param_reg__alpha":"alpha", "rmse_cv":"RMSE (CV)"}
)
fig.update_xaxes(type="log")
fig.show()